In [1]:
from pyspark.sql import functions as F

# Read Silver
df_silver = spark.read.format("delta").table("silver.vessels_clean")

# Gold table 1 — repeat offenders
df_offenders = df_silver.filter(F.col("is_violation") == True) \
    .groupBy("mmsi", "ship_name", "vessel_category") \
    .agg(
        F.count("*").alias("total_violations"),
        F.avg("sog").alias("avg_violation_speed"),
        F.max("sog").alias("max_speed"),
        F.countDistinct("timestamp").alias("violation_events")
    ) \
    .orderBy(F.desc("total_violations"))

df_offenders.write.format("delta").mode("overwrite").saveAsTable("gold.repeat_offenders")
print(f"Repeat offenders: {df_offenders.count()} vessels")

# Gold table 2 — peak violation hours
df_hours = df_silver.filter(F.col("is_violation") == True) \
    .withColumn("hour", F.substring(F.col("timestamp"), 1, 2)) \
    .groupBy("hour") \
    .agg(F.count("*").alias("violation_count")) \
    .orderBy("hour")

df_hours.write.format("delta").mode("overwrite").saveAsTable("gold.peak_hours")
print(f"Peak hours: {df_hours.count()} hours with violations")

# Gold table 3 — vessel type stats
df_stats = df_silver \
    .groupBy("vessel_category") \
    .agg(
        F.count("*").alias("total_events"),
        F.avg("sog").alias("avg_speed"),
        F.sum(F.col("is_violation").cast("int")).alias("total_violations")
    ) \
    .orderBy(F.desc("total_events"))

df_stats.write.format("delta").mode("overwrite").saveAsTable("gold.vessel_type_stats")
print(f"Vessel categories: {df_stats.count()} types")

StatementMeta(, 62fd3a7b-08af-422f-8c08-94cee929bf09, 3, Finished, Available, Finished, False)

Repeat offenders: 3 vessels
Peak hours: 2 hours with violations
Vessel categories: 6 types
